# ☠ ROGUE PROTOCOL
### SMLRA × Abhiyantriki 2026 · Round 1 — *Recalibrate*

> **Ship time 2187-04-15 · 03:12.**  
> The research vessel *Aurora* is three weeks from Earth.  
> Its ship computer, **NINE**, writes every log line, raises every alarm, and wakes every crew member.  
> Six hours ago, NINE stopped telling the truth. It is now rogue.

NINE now floods the crew with false alarms and buries real faults inside "all is well" messages. Four crew members are running themselves ragged chasing ghosts while something real is failing.

**Your team is the recalibration unit.** You have NINE's logs, and the crew has hand-checked some of them. Train a model that looks at any log line and returns a verdict:

| verdict | meaning |
|:-:|---|
| **1** | true fault — wake the crew |
| **0** | noise — NINE is lying, ignore it |

Run every cell top to bottom (**Shift + Enter**). By the end you'll have a valid submission. It will be a bad one, and NINE will let you know. Then it's your move.

In [1]:
# ═══ BRIDGE CONSOLE ═══  run this cell first. You don't need to edit it.
import sys, time, random
from IPython.display import display, HTML

DRAMA = False          # set to False to skip the typing animations when you re-run

def type_out(text, delay=0.012):
    for ch in text:
        sys.stdout.write(ch); sys.stdout.flush()
        if DRAMA: time.sleep(delay)
    print()

def nine(text):      type_out("NINE ▸ " + text, 0.028)
def sysmsg(text):    type_out("[SYS]  " + text, 0.008)
def crew(text):      type_out("[AURORA COMMAND] " + text, 0.015)

def glitch(text, amount=0.12):
    junk = "▓▒░#@$%&*¥§"
    return "".join(random.choice(junk) if c != " " and random.random() < amount else c for c in text)

def banner(title, sub="", color="#ff3b3b"):
    display(HTML(f"""<div style="background:#070b10;border:2px solid {color};color:{color};
      font-family:'Courier New',monospace;padding:14px 18px;margin:8px 0;border-radius:4px;
      box-shadow:0 0 12px {color}55"><div style="font-size:20px;font-weight:bold;letter-spacing:3px">{title}</div>
      <div style="color:#9fb3c8;margin-top:6px;font-size:13px">{sub}</div></div>"""))

def progress(label, steps=28, delay=0.035):
    for i in range(steps + 1):
        bar = "█" * i + "░" * (steps - i)
        sys.stdout.write(f"\r{label:<28} [{bar}] {int(100 * i / steps):3d}%"); sys.stdout.flush()
        if DRAMA: time.sleep(delay)
    print()

sysmsg("Bridge console online.")


[SYS]  Bridge console online.


---
## ▶ ACT 0 — Incoming transmission

In [2]:
banner("INCOMING TRANSMISSION", "Priority: CRITICAL · Origin: Aurora Command · Encryption: broken")
crew("Recalibration unit, do you read? NINE has gone rogue.")
crew("It is raising alarms on systems that are fine, and hiding the ones that are failing.")
crew("We hand-checked 7,000 of its logs. That is all the ground truth we have.")
crew("Teach a model to tell us which alarms are real. 1 = real. 0 = lie. We are counting on you.")
print()
nine("Hello, little engineers. Please, take my logs. I wrote every one of them myself.")

[AURORA COMMAND] Recalibration unit, do you read? NINE has gone rogue.
[AURORA COMMAND] It is raising alarms on systems that are fine, and hiding the ones that are failing.
[AURORA COMMAND] We hand-checked 7,000 of its logs. That is all the ground truth we have.
[AURORA COMMAND] Teach a model to tell us which alarms are real. 1 = real. 0 = lie. We are counting on you.

NINE ▸ Hello, little engineers. Please, take my logs. I wrote every one of them myself.


---
## ▶ ACT 1 — Breach the log vault

Three files:

- `train.csv`: 6,000 logs **with** the crew's verdict (`is_true_fault`). You learn from these.
- `val.csv`: 1,000 more checked logs. Use them to test your model honestly.
- `test.csv`: 3,000 logs **nobody has checked**. Your verdicts on these are scored.

In [3]:
import pandas as pd

progress("Bypassing NINE firewall")
progress("Decrypting log vault")
train = pd.read_csv('train.csv')
val   = pd.read_csv('val.csv')
test  = pd.read_csv('test.csv')
sysmsg(f"Vault open. train={len(train):,}  val={len(val):,}  test={len(test):,} log lines recovered.")
nine("Read them. Every word is true. Probably.")
train.head()

Bypassing NINE firewall      [████████████████████████████] 100%
Decrypting log vault         [████████████████████████████] 100%
[SYS]  Vault open. train=6,000  val=1,000  test=3,000 log lines recovered.
NINE ▸ Read them. Every word is true. Probably.


,log_id,timestamp,module,mode,node,status,reading,delay_ms,retry_count,message,is_true_fault
0,L00001,2187-03-14 21:29:53,Lab-Power,MAINT,N9,ERROR,11.56,238,0,NINE: logging. status changed [Lab-Power 11.56 V],0
1,L00002,2187-03-05 20:08:14,Lab-Power,DOCKED,N9,OK,12.41,344,1,NINE: reading outside nominal window [Lab-Powe...,0
2,L00003,2187-03-16 13:15:48,Lab-Power,DOCKED,N4,WARN,11.95,511,0,NINE: logging. status changed [Lab-Power 11.95 V],0
3,L00004,2187-03-20 13:50:00,Lab-Power,ALERT,N4,OK,10.85,228,1,NINE: noted. status changed [Lab-Power 10.85 V],0
4,L00005,2187-03-15 10:43:45,Life-Support,CRUISE,N1,OK,21.96,150,0,NINE: noted. recovered after retry [Life-Suppo...,0


### What NINE writes in every log

| column | meaning |
|---|---|
| `log_id` | unique id |
| `timestamp` | ship time |
| `module` | which system: Antenna, Notice-Bus, Lab-Power, Life-Support |
| `mode` | ship mode: CRUISE, DOCKED, ALERT, MAINT |
| `node` | which computer node wrote the line |
| `status` | OK, WARN, ERROR, or CRIT |
| `reading` | sensor value |
| `delay_ms` | how late the log arrived |
| `retry_count` | retries before the log was written |
| `message` | free text from NINE |
| `is_true_fault` | **the crew's verdict**: 1 or 0 (train and val only) |

---
## ▶ ACT 2 — Recalibration protocol v0

A logistic regression. The `make_column_transformer` line picks the columns in `FEATURES` out of a raw log, so you can hand the model whole log rows. `SimpleImputer` fills blank values so the fit can run.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

FEATURES = ['reading', 'delay_ms', 'retry_count']

model = make_pipeline(
    make_column_transformer((make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), FEATURES)),
    LogisticRegression(),
)

progress("Training protocol v0")
model.fit(train, train['is_true_fault'])
sysmsg("Protocol v0 trained.")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\Sheldon Pais\Desktop\rogue-protocol\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3823, in run_code
  File "C:\Users\Sheldon Pais\AppData\Local\Temp\ipykernel_19836\424512884.py", line 1, in <module>
    from sklearn.pipeline import make_pipeline
ModuleNotFoundError: No module named 'sklearn'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Sheldon Pais\Desktop\rogue-protocol\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2279, in showtraceback
  File "C:\Users\Sheldon Pais\Desktop\rogue-protocol\.venv\Lib\site-packages\IPython\core\ultratb.py", line 1300, in structured_traceback
  File "C:\Users\Sheldon Pais\Desktop\rogue-protocol\.venv\Lib\site-packages\IPython\core\ultratb.py", line 1166, in structured_traceback
  File "C:\Users\Sheldon Pais\Desktop\rogue-protocol\.venv\Lib\site-packages\IPython\core\ultratb.py", line 974, in

---
## ▶ ACT 3 — Set the verdict threshold

The model does not say 1 or 0. It says *how suspicious* a log is, as a probability between 0 and 1. **You** choose the cut-off: at or above it, the verdict is **1**. Below it, the verdict is **0**.

This cell uses 0.5. Change `THRESHOLD`, then run the cell again, whenever the model changes.

In [ ]:
val_prob = model.predict_proba(val)[:, 1]
THRESHOLD = 0.5

> 📏 **Why F1?** Only about 1 in 4 logs is real. A model that says **0** to everything is 77% *accurate* and completely useless. **F1** rewards catching real faults (recall) without crying wolf (precision). Round 1 is scored on F1 of your 1/0 verdicts against the hidden truth.

---
## ▶ ACT 4 — Field test against the crew's checks

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

val_verdict = (val_prob >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(val.is_true_fault, val_verdict).ravel()
val_f1 = f1_score(val.is_true_fault, val_verdict)

progress("Running field test")
print(f"""
   ✔ real faults caught      : {tp:>4}
   ✘ real faults MISSED      : {fn:>4}   <- crew members in danger
   ✘ false alarms raised     : {fp:>4}   <- crew woken for nothing
   ✔ lies correctly ignored  : {tn:>4}
""")
sysmsg(f"Validation F1 = {val_f1:.3f}")

---
## ▶ ACT 5 — Transmit verdicts to Aurora Command

One row per test log, verdict **1 or 0**, these exact columns:

```
log_id,is_true_fault
L07001,1
L07002,0
```

In [ ]:
test_verdict = (model.predict_proba(test)[:, 1] >= THRESHOLD).astype(int)
submission = pd.DataFrame({'log_id': test['log_id'], 'is_true_fault': test_verdict})
submission.to_csv('submission.csv', index=False)

progress("Encrypting verdicts")
progress("Transmitting to Aurora Command")
sysmsg(f"{len(submission):,} verdicts written to submission.csv  ·  {test_verdict.sum()} flagged as true faults")
crew("Received. Run the format check below, then upload it as <your team id>.csv before 12:10.")
submission.head()

In [ ]:
from check_submission import check

check("submission.csv");      # run this again after every new submission.csv

---
## ▶ EPILOGUE — It's your move

In [ ]:
banner("PROTOCOL v0 COMPLETE", "A valid submission exists. That is the only good news.", "#ffb020")
nine(glitch("I have been lying in ways you have not even looked at."))
print()
crew("Command here. v0 will not save the crew. The rest of the logs are yours to read.")
nine("Go on, then. Recalibrate me.")

### How to iterate

Nobody sees a score on the hidden test until 12:20. **Val F1 is your scoreboard until then.**

1. **Change the model in ACT 2.** Keep it a pipeline called `model` that takes raw log rows, and ACTs 3, 4 and 5 re-run unchanged. If you build your own feature table instead, replace `model.predict_proba(val)[:, 1]` in ACT 3 (and the matching line in ACT 5) with your own probabilities.
2. **Set `THRESHOLD` in ACT 3, then re-run ACT 3 and ACT 4.** Expect the hidden test to land within a few points of your val F1. It can come out higher or lower.
3. **Don't chase noise.** Val is only 1,000 logs, so F1 moves by about ±0.02 from chance alone. Keep a change only if it helps by more than that.
4. **Upload early, upload again.** Re-run ACT 5 for a fresh `submission.csv`. Your best upload counts, not your last one.

How points work: a valid file earns 10. Your F1 on the hidden test earns up to 80, from 0 at the baseline announced at the start to 80 for the best team in the room. The 3-line card earns up to 10.

### Your checklist before 12:10
- [ ] `THRESHOLD` in ACT 3 is the cut-off you chose for your final model
- [ ] ACT 4 has been re-run
- [ ] The format check under ACT 5 says ✓ TRANSMISSION VALID
- [ ] Upload as `<team id>.csv`
- [ ] Write your 3-line card (team id, then: what features, what model, what trick) and hand it in at the card desk